# Information Theory Diagnostics for Cellular Infrastructure

This notebook computes and visualizes information theory metrics on cellular telemetry from Airtel Rwanda. We investigate:
1. **Shannon Entropy** of diurnal traffic volumes to analyze network demand predictability.
2. **Kullback-Leibler (KL) Divergence** to compare traffic profile variations between weekdays and weekends.
3. **Mutual Information** between Radio Access Technology (RAT) generations and traffic types.
4. **RRC State space & Feedback States** estimation.
5. **Satellite RTT Wall & Channel Dispersion** calculation.
6. **Peak Age of Information (AoI) Optimization** over core signaling queues.
7. **Zipf-Mandelbrot Content Entropy & Edge Caching Limits** calculation.
8. **Effective Capacity** under delay constraints.
9. **Transfer Entropy (Causality)** from DNS requests to GTP-C sessions.
10. **TCP Congestion Window KL Divergence** to BDP targets.
11. **Hardware-Conditional Entropy** of signaling overhead by TAC class.
12. **Age of Incorrect Information (AoII)** optimization over CTMC RRC states.
13. **Goal-Oriented Private Semantic Caching** Pareto frontier optimization.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import fynesse
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 120

## 1. Data Ingestion
We load the aggregated temporal trace of TCP/UDP flows.

In [2]:
df = fynesse.load_joined_temporal_data()
df.head()

,time_idx,device,code,rat,val,len,Unnamed: 6,timestamp,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,hour,is_weekend,domain
0,395345,0,ut,0,0,0,NaN,2015-02-06 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,17,False,googlevideo
1,395345,0,dt,0,0,0,NaN,2015-02-06 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,17,False,googlevideo
2,395345,0,um,0,0,0,NaN,2015-02-06 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,17,False,googlevideo
3,395345,0,dm,0,0,0,NaN,2015-02-06 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,17,False,googlevideo
4,395345,0,ut,1,48899350,27363,NaN,2015-02-06 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,17,False,googlevideo


## 2. Diurnal Demand Predictability (Shannon Entropy)
We compute the Shannon Entropy $H(X)$ of hourly traffic volume distribution over the day:
$$H(X) = - \sum_{h=0}^{23} p_h \log_2 p_h$$

In [3]:
entropy = fynesse.calculate_hourly_traffic_entropy(df)

Computing Shannon Entropy of diurnal traffic load...
Generated hourly_traffic_entropy.png (Entropy = 4.3766 bits)


## 3. Weekday vs. Weekend Diurnal Divergence (Kullback-Leibler Divergence)
We measure the relative entropy difference between weekday hourly traffic distribution ($P$) and weekend hourly traffic distribution ($Q$):
$$D_{\text{KL}}(P \parallel Q) = \sum_{h=0}^{23} P_h \log_2 \left(\frac{P_h}{Q_h}\right)$$

In [4]:
kl_div = fynesse.calculate_weekday_weekend_kl_divergence(df)

Computing Kullback-Leibler (KL) Divergence between weekdays and weekends...
Generated weekday_weekend_kl_divergence.png (D_KL = 0.018924 bits)


## 4. RAT vs. Traffic Code (Mutual Information)
We evaluate how much information the choice of Radio Access Technology (RAT) shares with the type of traffic (uplink vs. downlink, payload vs. signaling):
$$I(RAT; Code) = \sum_{x \in RAT} \sum_{y \in Code} p(x,y) \log_2 \left(\frac{p(x,y)}{p(x)p(y)}\right)$$

In [5]:
mutual_info = fynesse.calculate_rat_traffic_code_mutual_information(df)

Computing Mutual Information between RAT and Traffic Code...
Generated rat_code_mutual_information.png (MI = 0.000034 bits)


## 5. RRC State space & Feedback States
We estimate the stationary distribution vector $\mathbf{P}_S$ of Dedicated, Shared, and Idle states:
$$\mathbf{P}_S = [\pi_D, \pi_S, \pi_I]$$

In [6]:
qos_df = fynesse.load_qos_metrics()
p_s = fynesse.estimate_feedback_channel_states(qos_df)

Estimating feedback channel RRC states from QoS telemetry...
Empirical State Distribution P_S:
  - Dedicated (Low Latency): 23.15%
  - Shared (RLC Reordering): 0.13%
  - Idle (Retransmissions):  76.72%
Generated rrc_feedback_states.png


## 6. Satellite RTT Wall & Channel Dispersion
We extract the RTT step-function above 200ms and compute the empirical channel dispersion $V_{\text{sat}}$ in seconds squared:
$$V_{\text{sat}} = \mathbb{E}[\text{Var}[\text{RTT} \mid X]]$$

In [7]:
wan_df = fynesse.load_rtt_data(sheet_name="wan")
mean_rtt, var_rtt = fynesse.estimate_satellite_dispersion(wan_df)

Estimating satellite channel dispersion V_sat from WAN latency data...
Satellite WAN Link Latency Summary (RTT >= 200ms):
  - Mean Satellite RTT: 367.13 ms
  - Empirical Dispersion V_sat: 2.428995 sec^2
Generated satellite_fbl_dispersion.png


## 7. Peak Age of Information (AoI) Optimization
We process core GTP-C request timelines to calculate arrival rate and solve the $M/GI/1/K$ queueing-theoretic model to locate the optimal arrival rate $\lambda^*$ that minimizes information staleness:
$$\mathbb{E}[\Delta_{\text{peak}}] = \mathbb{E}[T] + \frac{1}{\lambda_{\text{sig}} (1 - P_{\text{drop}})}$$

In [8]:
gtpc_df = fynesse.load_gtpc_signaling()
opt_lambda, opt_aoi = fynesse.estimate_core_signaling_aoi(gtpc_df)

Estimating core signaling parameters and Peak AoI optimization...
GTP-C Signaling Core Statistics:
  - Total Messages: 166538
  - Trace Duration: 1200.00 seconds
  - Empirical Arrival Rate (lambda_emp): 138.78 msgs/sec
Generated core_queue_aoi_optimization.png (Optimal lambda* = 112.70 msgs/sec)


## 8. Zipf-Mandelbrot Content Entropy & Edge Caching Limits
We model domain requests using the Zipf-Mandelbrot distribution:
$$P(r) = \frac{C}{(r + q)^\alpha}$$
We calculate the content request entropy:
$$H(\text{Content}) = - \sum_{r=1}^{N} P(r) \log_2 P(r)$$
And compute the maximum theoretical cache hit rate for a cache size of $K$ elements:
$$\eta(K) = \sum_{r=1}^{K} P(r)$$

In [9]:
entropy_c, eta10, eta50 = fynesse.calculate_zipf_mandelbrot_caching()

Executing Zipf-Mandelbrot Content Entropy & Caching Limits...
Generated fig4_zipf_caching.png


## 9. Effective Capacity under delay constraints
We evaluate the Effective Capacity $E_c(\theta)$ showing the maximum constant arrival rate supported under a statistical delay-bound QoS exponent $\theta$:
$$E_c(\theta) = - \frac{1}{\theta} \lim_{t \to \infty} \frac{1}{t} \ln \mathbb{E}\left[e^{-\theta S(t)}\right]$$

In [10]:
thetas, ec_2g, ec_3g = fynesse.calculate_effective_capacity()

Executing Effective Capacity under delay constraints...


c:\Users\SAMMY\Documents\dsail\research\cellular\rwanda\notebook\..\fynesse\infotheory.py:483: RuntimeWarning: divide by zero encountered in log
  val_3g = - (1.0 / theta) * np.log(np.mean(np.exp(-theta * s_3g)))


Generated fig5_effective_capacity.png


## 10. DNS to GTP-C Transfer Entropy causality
We compute the Transfer Entropy $T_{X \to Y}$ (from DNS query timeline $X_t$ to GTP-C requests timeline $Y_t$) to mathematically prove causality from DNS resolution to core session storms:
$$T_{X \to Y} = \sum_{t=1}^N H(Y_t \mid Y_1^{t-1}) - H(Y_t \mid Y_1^{t-1}, X_1^{t-1})$$

In [11]:
t_xy = fynesse.calculate_dns_gtpc_transfer_entropy()

Executing Transfer Entropy DNS to GTP-C...
Generated fig6_transfer_entropy.png


## 11. TCP window BDP starvation KL divergence
We compute the relative entropy $D_{\text{KL}}(P_{\text{win}} \parallel P_{\text{BDP}})$ to prove window starvation on resource-constrained cellular downlinks:
$$D_{\text{KL}}(P_{\text{win}} \parallel P_{\text{BDP}}) = \sum_{w} P_{\text{win}}(w) \log_2 \left( \frac{P_{\text{win}}(w)}{P_{\text{BDP}}(w)} \right)$$

In [12]:
kl_win = fynesse.calculate_tcp_window_bdp_starvation()

Executing TCP Congestion Window Starvation KL Divergence...
Generated fig7_tcp_starvation.png


## 12. Hardware-Conditional Entropy of Signaling Overhead
We calculate the conditional entropy $H(S \mid D)$ of signaling overhead ratio $S$ given the hardware class $D$:
$$H(S \mid D) = \sum_{d \in \mathcal{D}} P(D=d) H(S \mid D=d)$$

In [13]:
h_s, h_s_d = fynesse.calculate_hardware_conditional_entropy()

Executing Hardware-Conditional Entropy of Signaling Overhead...
Generated fig8_hardware_conditional_entropy.png


## 13. Age of Incorrect Information (AoII) CTMC Optimization
We model RRC state transitions as a Continuous-Time Markov Chain (CTMC) and optimize the update threshold $\tau$ to minimize the Age of Incorrect Information (AoII) at the core gateway:
$$A(t) = (t - t_0) \cdot \mathbb{I}(S(t) \neq \hat{S}(t))$$
The optimization problem is formulated as:
$$\min_{\tau > 0} \mathbb{E}[A(\tau)] \quad \text{s.t.} \quad \lambda_{\text{update}}(\tau) \leq \Lambda_{\text{max}}$$

In [14]:
opt_tau, opt_aoii, opt_rate = fynesse.calculate_aoii_rrc_optimization()

Executing Age of Incorrect Information (AoII) CTMC Optimization...
Generated fig9_aoii_optimization.png (Optimal tau* = 0.46 s, AoII = 22.96 ms)


## 14. Private Semantic Caching Pareto Frontier Optimization
We optimize local edge caching task completion utility under user privacy constraints, tracing the Pareto frontier between information leakage $\epsilon$ and semantic utility:
$$\max_{x} \sum_{f} P(f) V_f x_f \quad \text{s.t.} \quad I(X_{\text{cache}}; D) \leq \epsilon$$

In [15]:
eps_range, utility_vals = fynesse.calculate_private_semantic_caching()

Executing Goal-Oriented Semantic Caching under Privacy constraints...
Generated fig10_semantic_private_caching.png
